# 第104章 模型保存与批量推理

把预处理与模型作为单一制品保存，完成输入校验、批量推理和结果复核。


## 先解决一个小问题

围绕“模型保存与批量推理”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。把预处理与模型作为单一制品保存，完成输入校验、批量推理和结果复核。


## 这章为什么先学

这是“机器学习”建模主线中的第 104 章，重点放在“模型保存与批量推理”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：保存完整 Pipeline、加载模型并复现预测、校验列名与顺序


## 做完要留下什么

完成一份围绕“模型保存与批量推理”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 保存完整 Pipeline
- 加载模型并复现预测
- 校验列名与顺序
- 设计批量预测输出


## 核心概念

- 模型制品必须包含预处理
- 推理输入契约包括字段、类型和单位
- 序列化文件不能从不可信来源加载
- 版本、训练时间和特征清单应随制品记录


## 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import pickle

data = load_iris(as_frame=True); X, y=data.data, data.target
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)).fit(X, y)
payload = pickle.dumps(model)
restored = pickle.loads(payload)
print('制品字节数:', len(payload), '特征:', model.feature_names_in_.tolist())


## 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
import pandas as pd, numpy as np

batch = X.head(5).copy()
required = restored.feature_names_in_.tolist()
assert batch.columns.tolist()==required
output = batch.assign(prediction=restored.predict(batch), confidence=restored.predict_proba(batch).max(axis=1))
display(output)
print('重载预测一致:', np.array_equal(model.predict(batch), restored.predict(batch)))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 只保存估计器而漏掉预处理
- 不校验输入字段和单位
- 加载不可信 pickle
- 没有记录 sklearn 版本和训练数据版本


## 综合练习

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
practice_batch = X.tail(3).copy()
practice_output = pd.DataFrame({'prediction':restored.predict(practice_batch), 'confidence':restored.predict_proba(practice_batch).max(axis=1)})
display(practice_output)

# 自检
assert len(practice_output)==3
assert practice_output.confidence.between(0,1).all()


## 本章小结

把预处理与模型作为单一制品保存，完成输入校验、批量推理和结果复核。


### 你已经掌握

- 保存完整 Pipeline
- 加载模型并复现预测
- 校验列名与顺序
- 设计批量预测输出


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 数据与问题定义 | 先明确样本、特征、目标和验证方式，再训练模型。 | `pickle.dumps()`、`pickle.loads()`、`model.feature_names_in_.tolist()`、`.fit()` |
| 模型、公式与诊断 | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | `X.head()`、`restored.feature_names_in_.tolist()`、`batch.columns.tolist()`、`batch.assign()` |


### 需要注意

- 只保存估计器而漏掉预处理
- 不校验输入字段和单位
- 加载不可信 pickle
- 没有记录 sklearn 版本和训练数据版本


### 完成检查

- [ ] 能够保存完整 Pipeline
- [ ] 能够加载模型并复现预测
- [ ] 能够校验列名与顺序
- [ ] 能够设计批量预测输出


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
